**© Copyright AIDENTIFY. All rights reserved.**

# Part 2 | Session 15: Continuous Pretraining (지속 사전학습) with Qwen2.5-1.5B (LoRA)

## 🎯 Continuous Pretraining 개요

**Continuous Pretraining(지속 사전학습)**은 이미 사전 학습된 LLM에 도메인 특화 텍스트를 추가로 학습시키는 방법입니다.

### 왜 Continuous Pretraining이 필요한가?

- 🔹 사전학습 모델은 일반적인 지식은 풍부하지만, 특정 도메인 지식이 부족할 수 있습니다
- 🔹 의료, 법률, 금융 등 전문 분야의 용어와 지식을 모델에 주입할 수 있습니다
- 🔹 Instruction Tuning과 달리, **비구조화된 텍스트**(plain text)를 그대로 학습합니다
- 🔹 Next Token Prediction 방식으로 도메인 언어 패턴을 학습합니다

### 학습 목표

- ✅ Continuous Pretraining의 개념과 활용 사례 이해
- ✅ 도메인 텍스트 데이터 전처리 (토큰화 + 청크 분할)
- ✅ LoRA를 활용한 효율적 Continuous Pretraining 수행
- ✅ 학습 전후 도메인 지식 변화 비교

> 💡 **데이터 선택 포인트**: CPT의 효과를 분명히 보려면 모델이 **모르는** 지식을 써야 합니다.
> 이 실습은 Qwen이 사전학습으로 절대 접한 적 없는 **가상의 심해 도시 '메리디안'** 설정을
> 도메인 텍스트로 사용합니다. 일반 상식(예: AI/ML 개념)을 쓰면 모델이 이미 알고 있어
> 학습 전후 차이가 잘 드러나지 않습니다.

### 실습 환경

| 항목 | 설정 |
|------|------|
| 모델 | Qwen2.5-1.5B (4bit 양자화) |
| 방법 | LoRA (r=16) |
| GPU | RTX 4060 (8GB VRAM) |
| 데이터 | 가상의 도메인 텍스트 (심해 도시 '메리디안' 설정) |

## 1️⃣ 환경 설정 및 모델 로드 (Qwen2.5-1.5B, 4bit)

In [1]:
# 필수 라이브러리 임포트
import torch
import gc
import os
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 임포트 완료")
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/home/hpe/LLM_master_5parts/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Failed to load /home/hpe/LLM_master_5parts/venv/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/hpe/LLM_master_5parts/venv/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


Failed to load /home/hpe/LLM_master_5parts/venv/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/hpe/LLM_master_5parts/venv/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so


✅ 라이브러리 임포트 완료
PyTorch 버전: 2.11.0+cu128
CUDA 사용 가능: True
GPU: NVIDIA GeForce RTX 3070


In [2]:
# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

print_gpu_memory("시작")

[시작] GPU: 0.0GB / 7.6GB


In [3]:
# 모델 설정
MODEL_NAME = "Qwen/Qwen2.5-1.5B"  # Instruct 버전이 아닌 base 모델 사용
OUTPUT_DIR = "./output/continuous_pretraining"

# 4bit 양자화 설정 (RTX 4060 메모리 절약)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("✅ 양자화 설정 완료")
print(f"📌 모델: {MODEL_NAME}")
print(f"📌 양자화: 4bit NF4")

✅ 양자화 설정 완료
📌 모델: Qwen/Qwen2.5-1.5B
📌 양자화: 4bit NF4


In [4]:
# 토크나이저 로드
print("⏳ 토크나이저 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# pad_token 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ 토크나이저 로드 완료")
print(f"📌 Vocab 크기: {tokenizer.vocab_size:,}")
print(f"📌 EOS 토큰: {tokenizer.eos_token}")
print(f"📌 PAD 토큰: {tokenizer.pad_token}")

⏳ 토크나이저 로딩 중...


✅ 토크나이저 로드 완료
📌 Vocab 크기: 151,643
📌 EOS 토큰: <|endoftext|>
📌 PAD 토큰: <|endoftext|>


In [5]:
# 모델 로드 (4bit 양자화)
print("⏳ 모델 로딩 중... (4bit 양자화)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# gradient checkpointing 활성화 (메모리 절약)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

print("✅ 모델 로드 완료")
print(f"📌 모델 파라미터 수: {model.num_parameters():,}")
print_gpu_memory("모델 로드 후")

⏳ 모델 로딩 중... (4bit 양자화)


✅ 모델 로드 완료
📌 모델 파라미터 수: 1,543,714,304
[모델 로드 후] GPU: 1.1GB / 7.6GB


## 2️⃣ 도메인 텍스트 데이터 준비

Continuous Pretraining에서는 **비구조화된 텍스트**(plain text)를 사용합니다.

- 📄 Instruction-Output 쌍이 아닌 일반 텍스트
- 📄 도메인 특화 문서, 논문, 매뉴얼 등
- 📄 모델이 Next Token Prediction으로 도메인 언어를 학습

In [6]:
# 도메인 텍스트 데이터 로드
data_path = "../data/samples/domain_text_sample.txt"

print("⏳ 도메인 텍스트 데이터 로딩 중...")
with open(data_path, "r", encoding="utf-8") as f:
    domain_text = f.read()

print(f"✅ 데이터 로드 완료")
print(f"📌 전체 텍스트 길이: {len(domain_text):,} 글자")
print(f"📌 줄 수: {len(domain_text.splitlines())}줄")
print(f"\n--- 데이터 미리보기 (처음 500자) ---")
print(domain_text[:500])

⏳ 도메인 텍스트 데이터 로딩 중...
✅ 데이터 로드 완료
📌 전체 텍스트 길이: 2,714 글자
📌 줄 수: 61줄

--- 데이터 미리보기 (처음 500자) ---
심해 도시 메리디안 개요

메리디안(Meridian)은 2147년 태평양 마리아나 해구 인근 수심 3,200미터 지점에 건설된 세계 최초의 영구 거주형 심해 도시입니다. 메리디안은 해양공학자 리아 칸토스(Lia Kantos)가 이끄는 심해개척단에 의해 설계되었으며, 현재 약 12만 명의 시민이 거주하고 있습니다.

메리디안은 "심해는 인류의 마지막 개척지"라는 칸토스의 신념 아래 출발했습니다. 지상의 자원 고갈과 인구 과밀 문제를 해결하기 위한 대안으로 구상되었으며, 건설에는 총 18년이 걸렸습니다.

메리디안의 도시 구조

메리디안은 여섯 개의 거대한 압력 돔으로 이루어져 있습니다. 중앙에 위치한 코어 돔은 행정과 평의회가 자리한 핵심 구역이며, 할로우 돔은 주거 구역, 노바 돔은 연구 구역, 베르단트 돔은 식량 생산 구역, 포지 돔은 제조 구역, 그리고 게이트 돔은 외부 잠수정이 드나드는 항만 구역입니다.

여섯 개의 돔은 튜브라인(Tubeline)이라는 자기부상 교통망으로 연결


In [7]:
# 텍스트를 문단 단위로 분할
paragraphs = [p.strip() for p in domain_text.split("\n\n") if p.strip()]

print(f"✅ 문단 분할 완료")
print(f"📌 총 문단 수: {len(paragraphs)}")
print(f"\n--- 문단별 길이 ---")
for i, p in enumerate(paragraphs[:5]):
    print(f"  문단 {i+1}: {len(p)}자 - {p[:60]}...")

✅ 문단 분할 완료
📌 총 문단 수: 31

--- 문단별 길이 ---
  문단 1: 13자 - 심해 도시 메리디안 개요...
  문단 2: 160자 - 메리디안(Meridian)은 2147년 태평양 마리아나 해구 인근 수심 3,200미터 지점에 건설된 세계 최...
  문단 3: 105자 - 메리디안은 "심해는 인류의 마지막 개척지"라는 칸토스의 신념 아래 출발했습니다. 지상의 자원 고갈과 인구 과...
  문단 4: 11자 - 메리디안의 도시 구조...
  문단 5: 161자 - 메리디안은 여섯 개의 거대한 압력 돔으로 이루어져 있습니다. 중앙에 위치한 코어 돔은 행정과 평의회가 자리한...


## 3️⃣ 텍스트 데이터를 토큰화 및 청크 분할

Continuous Pretraining에서는 텍스트를 일정 길이의 **청크(chunk)**로 분할합니다.

- 🔹 max_seq_length 이내로 텍스트를 자릅니다
- 🔹 각 청크가 하나의 학습 샘플이 됩니다
- 🔹 Next Token Prediction: input_ids와 labels가 동일합니다

In [8]:
MAX_SEQ_LENGTH = 256  # 도메인 데이터가 적으므로 청크를 더 많이 만들어 학습 신호 확보

# 전체 텍스트를 하나로 합치고 토큰화
full_text = "\n\n".join(paragraphs)
print(f"⏳ 텍스트 토큰화 중...")

tokenized = tokenizer(
    full_text,
    return_tensors=None,
    add_special_tokens=False
)

all_input_ids = tokenized["input_ids"]
print(f"✅ 토큰화 완료")
print(f"📌 전체 토큰 수: {len(all_input_ids):,}")
print(f"📌 max_seq_length: {MAX_SEQ_LENGTH}")

⏳ 텍스트 토큰화 중...
✅ 토큰화 완료
📌 전체 토큰 수: 1,946
📌 max_seq_length: 256


In [9]:
# 토큰을 max_seq_length 크기의 청크로 분할
chunks = []
for i in range(0, len(all_input_ids), MAX_SEQ_LENGTH):
    chunk = all_input_ids[i:i + MAX_SEQ_LENGTH]
    if len(chunk) >= 64:  # 너무 짧은 청크는 제외
        chunks.append(chunk)

print(f"✅ 청크 분할 완료")
print(f"📌 총 청크 수: {len(chunks)}")
print(f"📌 각 청크 길이: {[len(c) for c in chunks]}")

✅ 청크 분할 완료
📌 총 청크 수: 8
📌 각 청크 길이: [256, 256, 256, 256, 256, 256, 256, 154]


In [10]:
# HuggingFace Dataset 형태로 변환
dataset_dict = {
    "input_ids": chunks,
    "attention_mask": [[1] * len(c) for c in chunks],
    "labels": chunks  # Causal LM: labels = input_ids
}

dataset = Dataset.from_dict(dataset_dict)

print(f"✅ Dataset 생성 완료")
print(f"📌 학습 샘플 수: {len(dataset)}")
print(f"📌 Dataset 컬럼: {dataset.column_names}")
print(f"\n--- 첫 번째 샘플 디코딩 (처음 200자) ---")
print(tokenizer.decode(dataset[0]["input_ids"][:100]))

✅ Dataset 생성 완료
📌 학습 샘플 수: 8
📌 Dataset 컬럼: ['input_ids', 'attention_mask', 'labels']

--- 첫 번째 샘플 디코딩 (처음 200자) ---
심해 도시 메리디안 개요

메리디안(Meridian)은 2147년 태평양 마리아나 해구 인근 수심 3,200미터 지점에 건설된 세계 최초의 영구 거주형 심해 도시입니다. 메리디안은 해양공학자 리아 칸토스(Lia Kantos)가 이끄는 심해개척


## 4️⃣ LoRA 설정 및 적용

LoRA를 사용하여 메모리 효율적으로 Continuous Pretraining을 수행합니다.

| 파라미터 | 값 | 설명 |
|----------|-----|------|
| r | 16 | LoRA 랭크 |
| lora_alpha | 32 | 스케일링 팩터 |
| target_modules | q_proj, k_proj, v_proj, o_proj | 어텐션 레이어 |
| task_type | CAUSAL_LM | Next Token Prediction |

In [11]:
# LoRA 설정
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print("✅ LoRA 설정 완료")
print(f"📌 r (랭크): {lora_config.r}")
print(f"📌 alpha: {lora_config.lora_alpha}")
print(f"📌 target_modules: {lora_config.target_modules}")
print(f"📌 dropout: {lora_config.lora_dropout}")

✅ LoRA 설정 완료
📌 r (랭크): 16
📌 alpha: 32
📌 target_modules: {'k_proj', 'v_proj', 'o_proj', 'q_proj'}
📌 dropout: 0.05


In [12]:
# LoRA 적용
print("⏳ LoRA 어댑터 적용 중...")
model = get_peft_model(model, lora_config)

# 학습 가능한 파라미터 확인
model.print_trainable_parameters()

print_gpu_memory("LoRA 적용 후")

⏳ LoRA 어댑터 적용 중...
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
[LoRA 적용 후] GPU: 1.1GB / 7.6GB


## 5️⃣ 학습 실행 (Trainer API)

RTX 4060에 안전한 학습 설정:

- 🔹 `per_device_train_batch_size=1` - 메모리 절약
- 🔹 `gradient_accumulation_steps=1` - 데이터가 적어 매 청크마다 업데이트
- 🔹 `bf16=True` - 혼합 정밀도 학습 (LoRA 파라미터는 fp32로 유지)
- 🔹 `num_train_epochs=10` - 도메인 데이터가 적으므로 여러 번 반복 학습

In [13]:
# 학습 전 모델 응답 저장 (비교용)
print("="*60)
print("📋 학습 전 모델 응답 (Before Training)")
print("="*60)

# 가상의 도메인(심해 도시 '메리디안')에 대한 질문.
# Qwen이 사전학습으로 알 수 없는 내용이므로, 학습 전에는 엉뚱하게 지어내고
# 학습 후에는 주입된 사실을 재현하는지 비교할 수 있습니다.
test_prompts = [
    "심해 도시 메리디안은",
    "메리디안을 설계하고 건설한 인물은",
    "메리디안의 모든 구조물에 사용된 아라곤 합금은",
    "메리디안의 주 에너지원인 청광 반응로는",
    "메리디안의 공식 화폐는"
]

model.eval()
before_responses = []

# ⚠️ 주의: 여기서 파라미터를 bf16으로 캐스팅하면 안 됩니다.
#    학습 대상인 LoRA 어댑터는 fp32로 유지되어야 옵티마이저 업데이트가 제대로 반영됩니다.
#    (bf16으로 바꾸면 작은 업데이트가 반올림으로 사라져 "학습이 안 되는" 현상이 발생)
#    4bit 베이스(bf16 출력) + fp32 LoRA의 dtype 불일치는 autocast로 해결합니다.
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    before_responses.append(response)
    print(f"\n🔹 프롬프트: {prompt}")
    print(f"   응답: {response[:200]}")

model.train()
print("\n" + "="*60)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


📋 학습 전 모델 응답 (Before Training)


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



🔹 프롬프트: 심해 도시 메리디안은
   응답: 심해 도시 메리디안은 2017년 1월 1일부터 2018년 12월 31일까지 1년간 100만원 이하의 저소득층이 100만원 이하의 저소득층이 100만원 이하의 저소득층이 100만원 이하의 저소득층이 100만원 이하의 저소득


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



🔹 프롬프트: 메리디안의 모든 구조물에 사용된 아라곤 합금은
   응답: 메리디안의 모든 구조물에 사용된 아라곤 합금은 100%의 완전한 품질을 보장합니다. 아라곤 합금은 100%의 완전한 품질을 보장합니다. 아라곤 합금은 100%의 완전한 품질을 보장합니다. 아라곤 합금은 100%의 완전한 품질을 보장합니다. 아라곤


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



🔹 프롬프트: 메리디안의 주 에너지원인 청광 반응로는
   응답: 메리디안의 주 에너지원인 청광 반응로는 2018년 1월 1일부터 2020년 12월 31일까지 2년간 1조 5천억원의 예산을 투입하여 건설된다. 청광 반응로는 2018년 1월 1일부터 2020년 12월 31일까지 2년간 1조 5천억원의



🔹 프롬프트: 메리디안의 공식 화폐는
   응답: 메리디안의 공식 화폐는 1960년에 발행되었습니다. 이 화폐는 1960년 1월 1일에 발행되었으며 1960년 1월 1일에 발행되었습니다. 이 화폐는 1960년 1월 1일에 발행되었습니다. 이 화폐는 1960년 1월 1일에 발행되었습니다. 이 화폐는 



In [14]:
# 학습 인자 설정
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=1,        # RTX 4060 안전 설정
    gradient_accumulation_steps=1,         # 데이터가 적어 매 청크마다 업데이트
    learning_rate=2e-4,
    bf16=True,                             # 혼합 정밀도
    logging_steps=1,
    save_strategy="epoch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
)

print("✅ TrainingArguments 설정 완료")
print(f"📌 batch_size: {training_args.per_device_train_batch_size}")
print(f"📌 gradient_accumulation: {training_args.gradient_accumulation_steps}")
print(f"📌 effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"📌 epochs: {training_args.num_train_epochs}")
print(f"📌 learning_rate: {training_args.learning_rate}")
print(f"📌 bf16: {training_args.bf16}")

✅ TrainingArguments 설정 완료
📌 batch_size: 1
📌 gradient_accumulation: 1
📌 effective batch size: 1
📌 epochs: 10
📌 learning_rate: 0.0002
📌 bf16: True


In [15]:
# Data Collator (Causal LM용 - MLM=False)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM이므로 MLM은 False
)

# Trainer 초기화
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

print("✅ Trainer 초기화 완료")
print_gpu_memory("학습 시작 전")

✅ Trainer 초기화 완료
[학습 시작 전] GPU: 1.1GB / 7.6GB


In [16]:
# 학습 실행
print("🚀 Continuous Pretraining 학습 시작!")
print("="*60)

train_result = trainer.train()

print("="*60)
print("✅ 학습 완료!")
print(f"📌 Total steps: {train_result.global_step}")
print(f"📌 Training loss: {train_result.training_loss:.4f}")
print_gpu_memory("학습 완료 후")

🚀 Continuous Pretraining 학습 시작!


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
1,2.375500
2,2.510000
3,2.196200
4,2.433200
5,2.439900
6,2.504500
7,2.671400
8,2.216100
9,2.591600
10,2.133600


✅ 학습 완료!
📌 Total steps: 80
📌 Training loss: 1.4192
[학습 완료 후] GPU: 1.1GB / 7.6GB


## 6️⃣ 학습 전후 비교 (도메인 지식 질문)

동일한 프롬프트로 학습 전후의 응답을 비교합니다.

- 🔹 도메인 용어에 대한 이해도 변화 확인
- 🔹 텍스트 생성 품질 비교
- 🔹 도메인 지식이 잘 주입되었는지 평가

In [17]:
# 학습 후 모델 응답 생성
print("="*60)
print("📋 학습 전후 비교 (Before vs After)")
print("="*60)

model.eval()
after_responses = []

# 학습 전(cell-19)과 동일하게, 파라미터는 그대로 두고 autocast로만 추론합니다.
for i, prompt in enumerate(test_prompts):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    after_responses.append(response)
    
    print(f"\n{'='*60}")
    print(f"🔹 프롬프트: {prompt}")
    print(f"\n  [Before] {before_responses[i][:200]}")
    print(f"\n  [After]  {response[:200]}")

print(f"\n{'='*60}")
print("📌 도메인 텍스트로 학습한 후, 관련 용어에 대한 연속 생성이 더 자연스러워졌는지 확인하세요.")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


📋 학습 전후 비교 (Before vs After)


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



🔹 프롬프트: 심해 도시 메리디안은

  [Before] 심해 도시 메리디안은 2017년 1월 1일부터 2018년 12월 31일까지 1년간 100만원 이하의 저소득층이 100만원 이하의 저소득층이 100만원 이하의 저소득층이 100만원 이하의 저소득층이 100만원 이하의 저소득

  [After]  심해 도시 메리디안은 매년 겨getQuery라 돔을 수심 300미터로 올리기 위해 임시로 도시를 확장합니다. 수심으로 열수를 조달하기 위해 열수 도시 아사곤과 측로 통로로形成的의 도시구조가 도厉됩니다.

메리디안은 매년 수많은 국제공헌금을 받으며, 특히 유해地区的 독립수珍공헌금


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



🔹 프롬프트: 메리디안의 모든 구조물에 사용된 아라곤 합금은

  [Before] 메리디안의 모든 구조물에 사용된 아라곤 합금은 100%의 완전한 품질을 보장합니다. 아라곤 합금은 100%의 완전한 품질을 보장합니다. 아라곤 합금은 100%의 완전한 품질을 보장합니다. 아라곤 합금은 100%의 완전한 품질을 보장합니다. 아라곤

  [After]  메리디안의 모든 구조물에 사용된 아라곤 합금은 독립 기관인 아라곤 심해 아라곤 합금 기관(Aragon Submarine Aragonium Committee)이 관리합니다. 아라곤 합금은 열수공압 시스템이 만들어지는 데, 이 시스템은 메리디안의 모든 구조물에 걸쳐あります. 아라곤 합금은 생명을 유지하는 데 매우 중요하며, 기관은 매년 아라곤 �


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



🔹 프롬프트: 메리디안의 주 에너지원인 청광 반응로는

  [Before] 메리디안의 주 에너지원인 청광 반응로는 2018년 1월 1일부터 2020년 12월 31일까지 2년간 1조 5천억원의 예산을 투입하여 건설된다. 청광 반응로는 2018년 1월 1일부터 2020년 12월 31일까지 2년간 1조 5천억원의

  [After]  메리디안의 주 에너지원인 청광 반응로는 에너지 공급의 핵심 기구입니다. 매년 수시로 평가되는 청광 반응로의 안전성은 시민들의 자치심과 열정으로 유지됩니다. 모든 청광 반응로 직원은 심해의 안전과 환경을 지키는 임무를 수행합니다.

메리디안의 심해 도시 구조

메리디안의 심해 도시



🔹 프롬프트: 메리디안의 공식 화폐는

  [Before] 메리디안의 공식 화폐는 1960년에 발행되었습니다. 이 화폐는 1960년 1월 1일에 발행되었으며 1960년 1월 1일에 발행되었습니다. 이 화폐는 1960년 1월 1일에 발행되었습니다. 이 화폐는 1960년 1월 1일에 발행되었습니다. 이 화폐는 

  [After]  메리디안의 공식 화폐는 리아스라(Lirasr)로, 돔의 기반이 되는 구형 돔으로 만드여. 리아스라의 급여는 매주 1회로 지급하며, 1 리아스라의 가치는 1 마리의 돔이 아닐까 싶이요.

메리디안의 경제는 매년 수시로 평가되며, 독립적인 경제조례위원회가

📌 도메인 텍스트로 학습한 후, 관련 용어에 대한 연속 생성이 더 자연스러워졌는지 확인하세요.


## 7️⃣ 모델 저장

학습된 LoRA 어댑터만 저장합니다. (원본 모델은 저장하지 않음)

In [18]:
# LoRA 어댑터 저장
save_path = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ 모델 저장 완료: {save_path}")

# 저장된 파일 크기 확인
total_size = 0
for root, dirs, files in os.walk(save_path):
    for f in files:
        fp = os.path.join(root, f)
        size = os.path.getsize(fp)
        total_size += size
        print(f"  📄 {f}: {size/1024/1024:.1f} MB")

print(f"\n📌 전체 어댑터 크기: {total_size/1024/1024:.1f} MB")
print(f"📌 원본 모델 크기 대비 매우 작은 크기입니다!")

✅ 모델 저장 완료: ./output/continuous_pretraining/lora_adapter
  📄 README.md: 0.0 MB
  📄 adapter_config.json: 0.0 MB
  📄 tokenizer_config.json: 0.0 MB
  📄 added_tokens.json: 0.0 MB
  📄 special_tokens_map.json: 0.0 MB
  📄 chat_template.jinja: 0.0 MB
  📄 adapter_model.safetensors: 16.7 MB
  📄 tokenizer.json: 10.9 MB
  📄 vocab.json: 2.6 MB
  📄 merges.txt: 1.6 MB

📌 전체 어댑터 크기: 31.8 MB
📌 원본 모델 크기 대비 매우 작은 크기입니다!


In [19]:
# GPU 메모리 정리
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print_gpu_memory("메모리 정리 후")
print("✅ GPU 메모리 정리 완료")

[메모리 정리 후] GPU: 0.0GB / 7.6GB
✅ GPU 메모리 정리 완료


## 📝 정리 및 핵심 요약

### 이번 실습에서 배운 내용

| 항목 | 내용 |
|------|------|
| Continuous Pretraining | 도메인 텍스트를 Next Token Prediction으로 추가 학습 |
| 데이터 준비 | 텍스트 → 토큰화 → 고정 길이 청크 분할 |
| LoRA 적용 | 4bit 양자화 + LoRA로 메모리 효율적 학습 |
| 학습 결과 | 도메인 용어와 개념에 대한 생성 품질 향상 |

### 핵심 포인트

- 🎯 **Continuous Pretraining은 비구조화 텍스트**를 사용합니다 (instruction 형식 아님)
- 🎯 **Next Token Prediction** 방식: labels = input_ids
- 🎯 **도메인 지식 주입**에 효과적 (의료, 법률, 금융 등)
- 🎯 **LoRA를 사용하면** 8GB VRAM으로도 수행 가능
- 🎯 실제 활용 시에는 **더 많은 도메인 데이터**가 필요합니다

### 다음 단계

- ➡️ **Session 16**: Instruction Tuning - 지시-응답 형식 데이터로 파인튜닝
- ➡️ Continuous Pretraining → Instruction Tuning 순서로 학습하면 더 효과적!